# 40 — Block 4: Hiển thị và lọc

- Lọc frame theo tiêu chí (thuộc tính cảnh, số box từng class, box nhỏ) bằng `DataFrame.query`.
- **Mẫu đang sai trước đó:** lỗi của base model trên dev (bỏ sót, nhầm class, FP) ở conf / IoU cố định → lọc và xem.
- Frame mà mỗi cách proposal đã chọn: vòng, thứ hạng, điểm từng tiêu chí, nhãn sau khi mở (chỉ ảnh đã chốt manifest).
- Sau đánh giá: lỗi nào của base đã được sửa, lỗi nào mới xuất hiện.

Ảnh pool chỉ xem được nhãn khi đã nằm trong một manifest đã chốt. Lỗi trên final test chỉ nên xem sau khi đánh giá xong.

**Input:** output notebook 10 (+ 20, 30 nếu có). **GPU:** nên có (inference base model).

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
SCENARIO = "missing"   # missing: base chưa có novel class | weak: base có ít ảnh novel
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"   # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
# luôn lấy bản mới nhất của COMMIT từ GitHub (kể cả khi /tmp/repo còn từ lần chạy trước trong session tương tác)
subprocess.run(["git", "-C", R, "fetch", "-q", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", "-f", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
for p in (f"{R}/src", R):   # `topicx` (harness) và `src.rav` (pipeline proposal, cùng đường import với P-026)
    if p not in sys.path:
        sys.path.insert(0, p)
for m in [m for m in sys.modules if m.split(".")[0] in ("topicx", "src")]:
    del sys.modules[m]   # chạy lại cell thì nạp lại code mới
import numpy as np, pandas as pd
from topicx import common as C, pipeline as P, proposals as PR
cfg = C.load_config(SMOKE, SCENARIO)
commit = subprocess.run(["git", "-C", R, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("code:", commit)
print(f"kịch bản={SCENARIO} | artifact={cfg['art']} | copy từ input: {C.sync_inputs(cfg)} file")

In [ ]:
STAGE = "explore"
# Test (CPU, vài giây): protocol của harness + test lõi P-026 + component mới. Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE);

## Health check dataset
Vài giây, không đọc cả file nhãn. Có mục FAIL thì notebook dừng ở đây.

In [ ]:
hc, sample = P.health_check(cfg)
if sample is not None:
    display(sample.resize((640, 360)))   # box đỏ phải ôm đúng vật thể

In [ ]:
from topicx import explore as X
NOVEL = C.load(C.WORK / cfg["art"] / "splits/meta.json")["novel"]
dev = X.frame_table(cfg, "dev")
display(dev.describe(include="all").T.head(20))

## Lọc frame theo tiêu chí

In [ ]:
QUERY = "timeofday == 'night' and n_{NOVEL} >= 1"   # sửa tuỳ ý, vd "weather == 'rainy' and n_small >= 5"
sub = dev.query(QUERY.format(NOVEL=NOVEL))
print(len(sub), "ảnh khớp:", QUERY.format(NOVEL=NOVEL))
pub = pd.read_csv(C.WORK / cfg["art"] / "splits/public_boxes.csv")
X.show(cfg, sub.image, gt=pub, title_cols=["timeofday", "weather", f"n_{NOVEL}"], frames=dev, n=8)

## Mẫu base model đang sai (dev)

In [ ]:
W = C.WORK / cfg["art"] / f"base/run/{cfg['yolo']['weights']}.pt"
gt, fp = X.model_errors(cfg, W, "dev", tag="base")
ef = X.error_frames(cfg, gt, fp, "dev")
display(X.recall_table(gt, ef, ["timeofday"]).pivot(index="cls", columns="timeofday", values="recall").round(3))
display(fp.kind.value_counts().head(10))

In [ ]:
ERR_QUERY = "timeofday == 'night' and miss_pedestrian >= 2"   # vd: người đi bộ ban đêm bị bỏ sót
bad = ef.query(ERR_QUERY).sort_values("n_miss", ascending=False)
print(len(bad), "ảnh khớp:", ERR_QUERY)
X.show(cfg, bad.image, gt=gt, fp=fp, title_cols=["n_miss", "n_fp", "timeofday", "weather"], frames=ef, n=8)   # vàng = bỏ sót, đỏ = FP

## Frame mỗi cách proposal đã chọn

In [ ]:
ex = PR.list_exports(cfg)
display(ex.drop(columns="path") if len(ex) else "chưa có export (chạy notebook 20 hoặc gắn output của nó)")

In [ ]:
NAME, SEED, K = (ex.iloc[0]["name"], int(ex.iloc[0]["seed"]), int(ex.iloc[0]["k"])) if len(ex) else (None, 0, 0)
if NAME and (C.WORK / cfg["art"] / f"selections/s{SEED}_k{K}_{NAME}.json").exists():   # manifest tạo ở notebook 30 (import_exports)
    sf, sbx = X.selection_frames(cfg, NAME, SEED, K)
    display(sf.head(20))
    display(sf.groupby("timeofday").size().rename("frames"))
    X.show(cfg, sf.image, gt=sbx, title_cols=["round", "rank", f"n_{NOVEL}"], frames=sf, n=8)
else:
    print("chưa có manifest đã chốt cho", NAME, "- chạy P.import_exports (notebook 30) trước")

## Sau đánh giá: lỗi của base đã được sửa (final test)

In [ ]:
RUN = None   # vd "s0_k256_algo1"; xem danh sách ở art/runs/
if RUN:
    gt_b, _ = X.model_errors(cfg, W, "test", tag="base")
    gt_r, fp_r = X.model_errors(cfg, C.WORK / cfg["art"] / f"runs/{RUN}/{cfg['yolo']['weights']}.pt", "test", tag=RUN)
    display(X.compare_errors(gt_b, gt_r))
    k = ["image", "cls", "x1", "y1", "x2", "y2"]
    m = gt_b[k + ["hit"]].merge(gt_r[k + ["hit"]], on=k, suffixes=("_b", "_r"))
    fixed = m[(m.cls == NOVEL) & ~m.hit_b & m.hit_r].image.unique()
    X.show(cfg, fixed, gt=gt_r, fp=fp_r, n=8)

## Gói kết quả
Zip file json/csv/md nhỏ (không có weights, embedding, nhãn oracle) → tab Output.

In [ ]:
P.bundle(cfg, STAGE)